In [1]:
# pip install requests beautifulsoup4 pandas

In [21]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin, quote_plus
import re
import random
from IPython.display import display  # for enhanced DataFrame display
from datetime import date

# --- User-Agent rotation ---
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 13_3_1) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.4 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:121.0) Gecko/20100101 Firefox/121.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 17_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2 Mobile/15E148 Safari/604.1",
]

# --- BBC Configuration ---
BBC_CONFIG = {
    "search_url": "https://www.bbc.co.uk/search?q={query}",
    "search_selector": "a.ssrcss-163mj99-PromoLink",
    "base_url": "https://www.bbc.co.uk"
}

# --- Scrape article details ---
def scrape_article_details(url):
    try:
        headers = {
            "User-Agent": random.choice(USER_AGENTS),
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.5",
            "Referer": "https://www.google.com/"
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # Headline
        headline = soup.title.text.strip() if soup.title else "N/A"
        if "N/A" in headline and soup.find('h1'):
            headline = soup.find('h1').text.strip()

        # Date
        date = "N/A"
        for meta in soup.find_all('meta', attrs={'property': re.compile(r'published|date', re.I)}):
            if 'content' in meta.attrs:
                date = meta['content'].strip()
                break

        # Content snippet
        content_snippet = "N/A"
        meta_desc = soup.find('meta', attrs={'name': 'description'})
        if meta_desc and 'content' in meta_desc.attrs:
            content_snippet = meta_desc['content'].strip()
        elif soup.find('p'):
            paragraphs = soup.find_all('p')
            content_snippet = ' '.join([p.text.strip() for p in paragraphs[:2] if p.text.strip()])[:200]

        # Extract first image
        image_url = "N/A"
        img_tag = soup.find('img')
        if img_tag and img_tag.get('src'):
            image_url = urljoin(url, img_tag.get('src'))

        return {
            "Source": "BBC",
            "Headline": headline,
            "Date": date,
            "URL": url,
            "Content_Snippet": content_snippet,
            "Image_URL": image_url
        }

    except Exception:
        return None

# --- Crawl BBC search results ---
def crawl_bbc(keyword):
    search_url = BBC_CONFIG["search_url"].format(query=quote_plus(keyword))
    selector = BBC_CONFIG["search_selector"]
    base_url = BBC_CONFIG["base_url"]

    print(f"\n---> Searching BBC ...")
    try:
        headers = {
            "User-Agent": random.choice(USER_AGENTS),
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.5",
            "Referer": "https://www.google.com/"
        }
        response = requests.get(search_url, headers=headers, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        links = soup.select(selector)

        urls = set()
        for link in links:
            href = link.get('href')
            if href:
                full_url = urljoin(base_url, href)
                urls.add(full_url)

        print(f"     ✅ Found {len(urls)} articles on BBC.")
        return list(urls)

    except Exception as e:
        print(f"     ❌ Failed BBC: {e}")
        return []

# --- Main ---
def main():
    print("\n--- Project Configuration ---")
    keyword = input("Enter the news keyword/topic: ").strip()
    if not keyword:
        print("Keyword cannot be empty. Exiting.")
        return

    urls = crawl_bbc(keyword)
    if not urls:
        print("❌ No articles found on BBC.")
        return

    print("\n--- Extracting Article Details ---")
    final_data = []
    total_articles = len(urls)
    print(f"Attempting to scrape {total_articles} articles...")

    for i, url in enumerate(urls, 1):
        print(f"  - ({i}/{total_articles}) Scraping: {url[:60]}...", end='\r')
        article_info = scrape_article_details(url)
        if article_info:
            final_data.append(article_info)

    print("\nExtraction complete.                     ")

    if final_data:
        df = pd.DataFrame(final_data)
        if "Date" in df.columns:
            df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        filename = f"News_Scraping_BBC_{keyword}_{pd.Timestamp('now').strftime('%Y%m%d_%H%M%S')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8')

        print(f"\n✅ Scraping Successful! {len(df)} articles saved to '{filename}'")
        print("\n--- First 5 Results ---")
        display(df.head())  # nicer display in Jupyter/Colab
    else:
        print("❌ No articles were scraped successfully.")

if __name__ == "__main__":
    main()



--- Project Configuration ---


Enter the news keyword/topic:  ai



---> Searching BBC ...
     ✅ Found 5 articles on BBC.

--- Extracting Article Details ---
Attempting to scrape 5 articles...
  - (5/5) Scraping: https://www.bbc.co.uk/sport/football/articles/cn82374p24go...
Extraction complete.                     

✅ Scraping Successful! 5 articles saved to 'News_Scraping_BBC_ai_20251016_212925.csv'

--- First 5 Results ---


C:\Users\DELL\AppData\Local\Temp\ipykernel_10900\2659037943.py:140: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


,Source,Headline,Date,URL,Content_Snippet,Image_URL
0,BBC,Artificial intelligence | Latest News & Update...,NaT,https://www.bbc.co.uk/news/topics/ce1qrvleleqt,Artificial Intelligence (AI) - get the latest ...,https://ichef.bbci.co.uk/ace/standard/480/cpsp...
1,BBC,Why AI is being trained in rural India - BBC News,NaT,https://www.bbc.co.uk/news/articles/cqjevxvxw9xo,Smaller Indian towns are becoming centres for ...,https://ichef.bbci.co.uk/ace/standard/768/cpsp...
2,BBC,Scottish data centres powering AI already usin...,NaT,https://www.bbc.co.uk/news/articles/c77zxx43x4vo,The volume of tap water used by Scotland's dat...,https://ichef.bbci.co.uk/ace/standard/2560/cps...
3,BBC,Dementia diagnosis revolutionised by AI and ne...,NaT,https://www.bbc.co.uk/news/articles/c62e5n8jlldo,Patients in Wales could be some of the first t...,https://ichef.bbci.co.uk/ace/standard/1800/cps...
4,BBC,Premier League predictions: Chris Sutton v Aya...,NaT,https://www.bbc.co.uk/sport/football/articles/...,BBC Sport football expert Chris Sutton takes o...,https://ichef.bbci.co.uk/ace/standard/2048/cps...


In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin, quote_plus
import re
import random
from IPython.display import display
import time

# --- User-Agent rotation ---
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 13_3_1) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.4 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:121.0) Gecko/20100101 Firefox/121.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 17_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2 Mobile/15E148 Safari/604.1",
]

# --- The Hindu Configuration ---
HINDU_CONFIG = {
    "search_url": "https://www.thehindu.com/search/?q={query}",
    "base_url": "https://www.thehindu.com",
    "search_selector": "a.story-card75x1-text, a.story-card75x1-text, a.story-card-img, a.story-card-link"
}


# --- Scrape article details ---
def scrape_hindu_article(url):
    try:
        headers = {
            "User-Agent": random.choice(USER_AGENTS),
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.5",
            "Referer": "https://www.google.com/"
        }

        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        # --- Headline ---
        headline = "N/A"
        if soup.title:
            headline = soup.title.text.strip()
        if "N/A" in headline and soup.find("h1"):
            headline = soup.find("h1").text.strip()

        # --- Date ---
        date = "N/A"
        date_meta = soup.find("meta", {"property": re.compile(r"published", re.I)})
        if date_meta and date_meta.get("content"):
            date = date_meta["content"].strip()

        # --- Author ---
        author = "N/A"
        author_meta = soup.find("meta", {"name": "author"})
        if author_meta and author_meta.get("content"):
            author = author_meta["content"].strip()
        elif soup.find("span", class_=re.compile(r"author", re.I)):
            author = soup.find("span", class_=re.compile(r"author", re.I)).get_text(strip=True)

        # --- Snippet ---
        snippet = "N/A"
        meta_desc = soup.find("meta", {"name": "description"})
        if meta_desc and meta_desc.get("content"):
            snippet = meta_desc["content"].strip()
        else:
            paras = soup.find_all("p")
            snippet = " ".join([p.text.strip() for p in paras[:2] if p.text.strip()])[:200] or "N/A"

        # --- Image ---
        image_url = "N/A"
        img = soup.find("img")
        if img and img.get("src"):
            image_url = urljoin(url, img.get("src"))

        return {
            "Source": "The Hindu",
            "Headline": headline,
            "Date": date,
            "Author": author,
            "URL": url,
            "Content_Snippet": snippet,
            "Image_URL": image_url
        }

    except Exception as e:
        # print(f"Failed article: {e}")
        return None


# --- Crawl Hindu search results ---
def crawl_hindu(keyword):
    search_url = HINDU_CONFIG["search_url"].format(query=quote_plus(keyword))
    selector = HINDU_CONFIG["search_selector"]
    base_url = HINDU_CONFIG["base_url"]

    print(f"\n---> Searching The Hindu ...")
    try:
        headers = {
            "User-Agent": random.choice(USER_AGENTS),
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.5",
            "Referer": "https://www.google.com/"
        }
        response = requests.get(search_url, headers=headers, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        # --- Collect links ---
        links = set()
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if re.search(r"/news/|/sport/|/entertainment/|/business/|/article\d+\.ece", href):
                full_url = urljoin(base_url, href)
                links.add(full_url)

        print(f"     ✅ Found {len(links)} article links on The Hindu.")
        return list(links)

    except Exception as e:
        print(f"     ❌ Failed The Hindu: {e}")
        return []


# --- Main ---
def main():
    print("\n--- The Hindu News Scraper ---")
    keyword = input("Enter the news keyword/topic: ").strip()
    if not keyword:
        print("Keyword cannot be empty. Exiting.")
        return

    urls = crawl_hindu(keyword)
    if not urls:
        print("❌ No articles found on The Hindu.")
        return

    print("\n--- Extracting Article Details ---")
    final_data = []
    total_articles = len(urls)
    print(f"Attempting to scrape {total_articles} articles...")

    for i, url in enumerate(urls, 1):
        print(f"  - ({i}/{total_articles}) Scraping: {url[:70]}...", end='\r')
        article_data = scrape_hindu_article(url)
        if article_data:
            final_data.append(article_data)
        time.sleep(0.7)  # polite delay

    print("\nExtraction complete.                     ")

    if final_data:
        df = pd.DataFrame(final_data)
        if "Date" in df.columns:
            df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

        filename = f"News_Scraping_TheHindu_{keyword}_{pd.Timestamp('now').strftime('%Y%m%d_%H%M%S')}.csv"
        df.to_csv(filename, index=False, encoding="utf-8")

        print(f"\n✅ Scraping Successful! {len(df)} articles saved to '{filename}'")
        print("\n--- First 5 Results ---")
        display(df.head())
    else:
        print("❌ No articles were scraped successfully.")


if __name__ == "__main__":
    main()



--- The Hindu News Scraper ---


Enter the news keyword/topic:  artificial intelligence



---> Searching The Hindu ...
     ✅ Found 63 article links on The Hindu.

--- Extracting Article Details ---
Attempting to scrape 63 articles...
  - (63/63) Scraping: https://www.thehindu.com/business/...Vijayawada/.../...n-live-score-as...
Extraction complete.                     

✅ Scraping Successful! 63 articles saved to 'News_Scraping_TheHindu_artificial intelligence_20251016_194606.csv'

--- First 5 Results ---


C:\Users\DELL\AppData\Local\Temp\ipykernel_10900\322963799.py:156: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


,Source,Headline,Date,Author,URL,Content_Snippet,Image_URL
0,The Hindu,"Movies | Latest News, Reviews, and Updates - T...",NaT,N/A,https://www.thehindu.com/entertainment/movies/,"Stay updated with the latest movie news, revie...",https://sb.scorecardresearch.com/p?c1=2&c2=113...
1,The Hindu,"Dance News, Dance Reviews, Classical Dance, Bh...",NaT,N/A,https://www.thehindu.com/entertainment/dance/,Follow News And Updates From The World Of Danc...,https://sb.scorecardresearch.com/p?c1=2&c2=113...
2,The Hindu,"Football News, Football Scores, Premier League...",NaT,N/A,https://www.thehindu.com/sport/football/,"Get Football News, Updates, Fixtures, Commenta...",https://sb.scorecardresearch.com/p?c1=2&c2=113...
3,The Hindu,IND vs ENG Tests in England: Full list of Engl...,2025-06-20 03:16:05+00:00,N/A,https://sportstar.thehindu.com/cricket/india-v...,India will look to clinch its first Test serie...,https://sb.scorecardresearch.com/p?c1=2&c2=113...
4,The Hindu,Hyderabad Latest News: Today’s Local Edition o...,NaT,N/A,https://www.thehindu.com/news/cities/Hyderabad/,Hyderabad News: Stay updated with the latest n...,https://sb.scorecardresearch.com/p?c1=2&c2=113...


In [55]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from urllib.parse import urljoin, quote_plus
import random
import re

# --- Step 1: Setup ---
HEADERS = {
    "User-Agent": random.choice([
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119 Safari/537.36",
    ])
}
BASE_URL = "https://www.thehindu.com"
SEARCH_URL = "https://www.thehindu.com/search/?q={query}&order=DESC&sort=publishdate"

# --- Step 2: Search for articles ---
def get_article_links(keyword, max_pages=1):
    all_links = []
    for page in range(1, max_pages+1):
        url = SEARCH_URL.format(query=quote_plus(keyword))
        print(f"🔍 Searching page {page}: {url}")
        res = requests.get(url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(res.text, "html.parser")

        # Each article link is usually inside 'div.story-card75x1-text a'
        for a in soup.select("div.story-card75x1-text a"):
            href = a.get("href")
            if href and href.startswith("http"):
                all_links.append(href)
        time.sleep(2)
    return list(set(all_links))

# --- Step 3: Scrape each article ---
def scrape_article(url):
    try:
        res = requests.get(url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(res.text, "html.parser")

        title = soup.find("h1").get_text(strip=True) if soup.find("h1") else None
        date_tag = soup.find("meta", {"property": "article:published_time"})
        publish_date = date_tag["content"] if date_tag else None

        # Extract article body text
        paragraphs = [p.get_text(strip=True) for p in soup.select("article p")]
        text = " ".join(paragraphs)

        # Extract first image (if any)
        img_tag = soup.find("img")
        image_url = img_tag["src"] if img_tag and img_tag.get("src") else None

        return {
            "title": title,
            "url": url,
            "publish_date": publish_date,
            "image_url": image_url,
            "content": text[:800] + "..." if len(text) > 800 else text,
        }
    except Exception as e:
        print("❌ Error scraping", url, "->", e)
        return None

# --- Step 4: Main driver ---
def scrape_the_hindu(keyword, pages=1):
    links = get_article_links(keyword, pages)
    print(f"✅ Found {len(links)} articles for '{keyword}'")
    records = []
    for link in links[:10]:  # limit to first 10 for safety
        data = scrape_article(link)
        if data:
            records.append(data)
        time.sleep(2)
    return pd.DataFrame(records)

# --- Step 5: Run & Display ---
df = scrape_the_hindu("artificial intelligence", pages=1)
print(df.head())
df.to_csv("the_hindu_articles.csv", index=False)


🔍 Searching page 1: https://www.thehindu.com/search/?q=artificial+intelligence&order=DESC&sort=publishdate
✅ Found 0 articles for 'artificial intelligence'
Empty DataFrame
Columns: []
Index: []


In [57]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from urllib.parse import quote_plus, urljoin
import random

# --- Step 1: Config ---
HEADERS = {
    "User-Agent": random.choice([
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119 Safari/537.36",
    ])
}
BASE_URL = "https://www.ndtv.com"
SEARCH_URL = "https://www.ndtv.com/search?searchtext={query}&page={page}"

# --- Step 2: Search for articles ---
def get_article_links(keyword, max_pages=1):
    links = []
    for page in range(1, max_pages + 1):
        search_url = SEARCH_URL.format(query=quote_plus(keyword), page=page)
        print(f"🔍 Searching NDTV page {page}: {search_url}")
        res = requests.get(search_url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(res.text, "html.parser")

        # Each search result card contains an <a> tag inside .src_lst li
        for a in soup.select("li.src_lst li a"):
            href = a.get("href")
            if href and href.startswith("http") and "ndtv.com" in href:
                links.append(href)

        time.sleep(2)
    return list(dict.fromkeys(links))  # remove duplicates

# --- Step 3: Scrape each article ---
def scrape_article(url):
    try:
        res = requests.get(url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(res.text, "html.parser")

        # Title
        title_tag = soup.find("h1") or soup.find("title")
        title = title_tag.get_text(strip=True) if title_tag else None

        # Date
        date_meta = soup.find("meta", {"property": "article:published_time"})
        publish_date = date_meta["content"] if date_meta else None

        # Body text (usually inside <div class="story__content">)
        paragraphs = [p.get_text(strip=True) for p in soup.select("div.story__content p")]
        if not paragraphs:
            paragraphs = [p.get_text(strip=True) for p in soup.find_all("p")]
        text = " ".join(paragraphs)

        # Image (main image)
        img_tag = soup.find("img")
        image_url = img_tag["src"] if img_tag and img_tag.get("src") else None

        return {
            "title": title,
            "url": url,
            "publish_date": publish_date,
            "image_url": image_url,
            "content": text[:800] + "..." if len(text) > 800 else text
        }

    except Exception as e:
        print("❌ Error scraping:", url, "->", e)
        return None

# --- Step 4: Main scraper function ---
def scrape_ndtv(keyword, pages=1):
    article_links = get_article_links(keyword, max_pages=pages)
    print(f"✅ Found {len(article_links)} articles for '{keyword}'")
    records = []
    for link in article_links[:10]:
        data = scrape_article(link)
        if data:
            records.append(data)
        time.sleep(2)
    return pd.DataFrame(records)

# --- Step 5: Run & Display ---
if __name__ == "__main__":
    df = scrape_ndtv("artificial intelligence", pages=1)
    print(df.head())
    df.to_csv("ndtv_articles.csv", index=False)


🔍 Searching NDTV page 1: https://www.ndtv.com/search?searchtext=artificial+intelligence&page=1
✅ Found 0 articles for 'artificial intelligence'
Empty DataFrame
Columns: []
Index: []


In [61]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from urllib.parse import quote_plus, urljoin
import random

# --- Step 1: Setup ---
HEADERS = {
    "User-Agent": random.choice([
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119 Safari/537.36",
    ])
}
BASE_URL = "https://www.indiatoday.in"
SEARCH_URL = "https://www.indiatoday.in/search/{query}&page={page}"

# --- Step 2: Collect article links ---
def get_article_links(keyword, max_pages=1):
    links = []
    for page in range(1, max_pages + 1):
        search_url = SEARCH_URL.format(query=quote_plus(keyword), page=page)
        print(f"🔍 Searching India Today page {page}: {search_url}")
        res = requests.get(search_url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(res.text, "html.parser")

        # Each search result is within div.catagory-listing a
        for a in soup.select("div.catagory-listing a"):
            href = a.get("href")
            if href and href.startswith("/"):
                links.append(urljoin(BASE_URL, href))
            elif href and "indiatoday.in" in href:
                links.append(href)
        time.sleep(2)
    return list(dict.fromkeys(links))   # remove duplicates

# --- Step 3: Scrape article details ---
def scrape_article(url):
    try:
        res = requests.get(url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(res.text, "html.parser")

        # Title
        title_tag = soup.find("h1") or soup.find("title")
        title = title_tag.get_text(strip=True) if title_tag else None

        # Date
        date_tag = soup.find("span", {"itemprop": "datePublished"}) or soup.find("meta", {"name": "publish-date"})
        publish_date = date_tag.get("content") if date_tag and date_tag.has_attr("content") else date_tag.get_text(strip=True) if date_tag else None

        # Article text
        paragraphs = [p.get_text(strip=True) for p in soup.select("div.description p")]
        if not paragraphs:
            paragraphs = [p.get_text(strip=True) for p in soup.find_all("p")]
        text = " ".join(paragraphs)

        # Image
        img_tag = soup.find("img")
        image_url = img_tag["src"] if img_tag and img_tag.get("src") else None

        return {
            "title": title,
            "url": url,
            "publish_date": publish_date,
            "image_url": image_url,
            "content": text[:800] + "..." if len(text) > 800 else text
        }

    except Exception as e:
        print("❌ Error scraping:", url, "→", e)
        return None

# --- Step 4: Combine into main function ---
def scrape_india_today(keyword, pages=1):
    article_links = get_article_links(keyword, max_pages=pages)
    print(f"✅ Found {len(article_links)} articles for '{keyword}'")
    records = []
    for link in article_links[:10]:
        data = scrape_article(link)
        if data:
            records.append(data)
        time.sleep(2)
    return pd.DataFrame(records)

# --- Step 5: Run ---
if __name__ == "__main__":
    df = scrape_india_today("artificial intelligence", pages=1)
    print(df.head())
    df.to_csv("india_today_articles.csv", index=False)


🔍 Searching India Today page 1: https://www.indiatoday.in/search/artificial+intelligence&page=1
✅ Found 0 articles for 'artificial intelligence'
Empty DataFrame
Columns: []
Index: []


In [63]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import quote_plus, urljoin
import re
import random

# --- User-Agent rotation ---
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)",
    "Mozilla/5.0 (X11; Linux x86_64)",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 14_0 like Mac OS X)",
    "Mozilla/5.0 (Windows NT 6.1; WOW64)",
]

def scrape_indianexpress(keyword, max_articles=10):
    print(f"🔍 Searching The Indian Express for: {keyword}")
    base_url = "https://indianexpress.com"
    search_url = f"{base_url}/?s={quote_plus(keyword)}"
    headers = {"User-Agent": random.choice(USER_AGENTS)}
    
    response = requests.get(search_url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    
    articles = []
    for item in soup.select("div.articles")[:max_articles]:
        title_tag = item.find("a")
        if not title_tag:
            continue
        title = title_tag.get_text(strip=True)
        link = title_tag["href"]
        summary = item.select_one(".snippets")
        summary = summary.get_text(strip=True) if summary else ""
        date = item.select_one(".date")
        date = date.get_text(strip=True) if date else ""
        
        # --- Fetch full article content ---
        article_resp = requests.get(link, headers=headers)
        article_soup = BeautifulSoup(article_resp.text, "html.parser")
        content = " ".join([p.get_text(strip=True) for p in article_soup.select("div.full-details p")])
        if not content:
            content = " ".join([p.get_text(strip=True) for p in article_soup.select("article p")])
        
        # --- Extract image ---
        img_tag = article_soup.find("img")
        img_url = urljoin(base_url, img_tag["src"]) if img_tag and img_tag.get("src") else ""
        
        articles.append({
            "Title": title,
            "URL": link,
            "Date": date,
            "Summary": summary,
            "Content": content[:500] + "...",
            "Image": img_url
        })
    
    df = pd.DataFrame(articles)
    display(df)
    return df


# Example usage
df_ie = scrape_indianexpress("AI", max_articles=5)


🔍 Searching The Indian Express for: AI


""
